# 07. Frozen-model external evaluation — separate ElasticNet and XGBoost runs

This paper-ready notebook applies the **prespecified** CCLE-trained models to all 20 external cohorts without external model selection or refitting.

The two multigene families are deliberately executed in **separate cells**:

1. prepare/reuse the shared disk-backed validation workspace;
2. run or resume **ElasticNet only**, with a live `tqdm` protein progress bar;
3. run or resume **XGBoost only**, with an independent progress bar;
4. finalize the portable cohort checkpoints and compact metrics only after both inventories are complete.

The same memory-safe design is retained: the external matrices and prediction matrices remain disk-backed, one HDF5 model is loaded at a time, predictions are generated in bounded sample batches, and no long sample-by-protein table is materialized. Progress is checkpointed to `state.json`, so interrupting either model family does not discard completed proteins.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "7_inference_all_models"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import gc
import json
import os
import sys

# Bound BLAS/OpenMP/XGBoost threading before importing numerical libraries.
for _thread_var in (
    "OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
    "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"
):
    os.environ.setdefault(_thread_var, "1")

# Find paper_validation.py beside the notebook/package or through an explicit override.
search_roots = []
if os.environ.get("PAPER_VALIDATION_MODULE_DIR"):
    search_roots.append(Path(os.environ["PAPER_VALIDATION_MODULE_DIR"]).expanduser().resolve())
cwd = Path.cwd().resolve()
for base in [cwd, *cwd.parents]:
    for candidate in [base, base / "paper_ready_external_validation", base / "notebooks", base / "notebooks2"]:
        if candidate not in search_roots:
            search_roots.append(candidate)
for candidate in search_roots:
    if (candidate / "paper_validation.py").exists():
        sys.path.insert(0, str(candidate))
        MODULE_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "paper_validation.py was not found. Keep the updated module beside this notebook "
        "or set PAPER_VALIDATION_MODULE_DIR."
    )

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
import paper_validation as pv

PROJECT_ROOT = pv.detect_project_root()
RUN_ROOT = Path(os.environ.get(
    "PAPER_VALIDATION_RUN_ROOT",
    PROJECT_ROOT / "results_external_validation_paper_v1",
)).expanduser().resolve()
LAYOUT = pv.make_output_layout(RUN_ROOT)
MANIFEST_PATH = RUN_ROOT / "validation_manifest.json"
SMOKE_TEST = os.environ.get("PAPER_VALIDATION_SMOKE_TEST", "0") == "1"

print("Notebook 07 — separate model-family execution")
print("Shared module:", MODULE_DIR / "paper_validation.py")
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Smoke-test mode:", SMOKE_TEST)

## 1. Load and validate the frozen manifest

Notebook 06 must have completed successfully. This cell verifies the frozen reference, proxy parameters, and exact model inventory before any external prediction is made.

In [ ]:
manifest = pv.load_manifest(MANIFEST_PATH)
MANIFEST_HASH = manifest["manifest_hash"]

if manifest["primary_analysis"]["external_model_selection"] is not False:
    raise RuntimeError("The frozen manifest permits external model selection, which is not paper-ready.")
if manifest["primary_analysis"]["external_sctranslator"] != "removed_incomplete":
    raise RuntimeError("External scTranslator must be complete or explicitly removed.")

for label in ["ccle_reference", "linear_proxy_parameters", "model_inventory"]:
    spec = manifest["files"][label]
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Frozen contract file is missing or changed: {label} -> {path}")

for cohort, spec in manifest["files"]["cohort_contracts"].items():
    path = Path(spec["path"])
    if not path.exists() or pv.file_sha256(path) != spec["sha256"]:
        raise RuntimeError(f"Cohort contract is missing or changed: {cohort} -> {path}")

reference = joblib.load(manifest["files"]["ccle_reference"]["path"])
proxy_parameters = pv.read_table(manifest["files"]["linear_proxy_parameters"]["path"])
model_inventory = pv.read_table(manifest["files"]["model_inventory"]["path"])

inventory_hash = pv.stable_hash(
    model_inventory.sort_values(["display_model", "protein"]).to_dict("records")
)
if inventory_hash != manifest["model_inventory_hash"]:
    raise RuntimeError("The loaded primary model inventory does not match the manifest hash.")

inventory_summary = (
    model_inventory.groupby("display_model", as_index=False)
    .agg(proteins=("protein", "nunique"), files=("path", "size"))
)
print("Manifest hash:", MANIFEST_HASH)
display(inventory_summary)

## 2. Execution configuration

`RESET_ALL=True` removes the shared evaluation workspace and invalidates previously finalized cohort checkpoints before the ElasticNet cell starts. Leave it `False` to resume.

The progress bar advances **one protein model at a time**. Checkpoints are also flushed periodically, so the bar can be interrupted and resumed without starting from protein 1.

In [ ]:
RESET_ALL = os.environ.get("PAPER_VALIDATION_FORCE", "0") == "1"
COMPUTE_SPEARMAN = os.environ.get("PAPER_VALIDATION_SPEARMAN", "1") == "1"
COMPRESSED_NPZ = os.environ.get("PAPER_VALIDATION_COMPRESS_NPZ", "1") == "1"
SAMPLE_BATCH_SIZE = int(os.environ.get(
    "PAPER_VALIDATION_SAMPLE_BATCH", "128" if SMOKE_TEST else "512"
))
CHECKPOINT_EVERY = int(os.environ.get(
    "PAPER_VALIDATION_CHECKPOINT_EVERY", "2" if SMOKE_TEST else "50"
))
# Keep the workspace after finalization by default so either family can be inspected/resumed.
CLEANUP_WORK_AFTER_FINALIZE = os.environ.get("PAPER_VALIDATION_CLEANUP_WORK", "0") == "1"

cohort_bundle_paths = {
    cohort: spec["path"]
    for cohort, spec in manifest["files"]["cohort_contracts"].items()
}
work_dir = LAYOUT["evaluation"] / f"work_{MANIFEST_HASH[:16]}"
state_path = work_dir / "state.json"

def show_family_state():
    if not state_path.exists():
        return pd.DataFrame({
            "display_model": ["ElasticNet", "XGBoost"],
            "completed_proteins": [0, 0],
            "total_proteins": [
                int((model_inventory["display_model"] == "ElasticNet").sum()),
                int((model_inventory["display_model"] == "XGBoost").sum()),
            ],
        })
    state = json.loads(state_path.read_text(encoding="utf-8"))
    rows = []
    for family in ["ElasticNet", "XGBoost"]:
        total = int((model_inventory["display_model"] == family).sum())
        done = int(state.get("progress", {}).get(family, 0))
        rows.append({
            "display_model": family,
            "completed_proteins": done,
            "total_proteins": total,
            "percent": 100.0 * done / total if total else np.nan,
        })
    return pd.DataFrame(rows)

print("RESET_ALL:", RESET_ALL)
print("Sample batch size:", SAMPLE_BATCH_SIZE)
print("Checkpoint every proteins:", CHECKPOINT_EVERY)
print("Work directory:", work_dir)
display(show_family_state())

## 3. Run / resume ElasticNet only

Run this cell independently. It evaluates only the prespecified z-score, \(k=5000\) ElasticNet inventory. XGBoost is not evaluated in this cell.

In [ ]:
enet_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=RESET_ALL,
    cleanup_work_on_success=False,
    model_families=("ElasticNet",),
    show_progress=True,
    finalize=False,
)
RESET_ALL = False  # Never erase the completed ENet run when the XGB cell is executed next.
gc.collect()
display(enet_progress)
display(show_family_state())

## 4. Run / resume XGBoost only

This cell uses the same disk-backed RNA inputs and workspace but evaluates **only XGBoost**. It can be executed later or resumed independently after interruption.

In [ ]:
xgb_progress = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=False,
    model_families=("XGBoost",),
    show_progress=True,
    finalize=False,
)
gc.collect()
display(xgb_progress)
display(show_family_state())

## 5. Finalize the 20 cohort checkpoints

This cell does **not** rerun either model family. It first requires both progress counters to equal their complete model inventories, then writes the per-cohort NPZ prediction checkpoints, compact metric partitions, transformation audits, and completion-status files consumed by notebooks 08–10.

In [ ]:
completion = pv.evaluate_all_cohorts_memory_safe(
    cohort_bundle_paths,
    reference,
    proxy_parameters,
    model_inventory,
    manifest_hash_value=MANIFEST_HASH,
    prediction_dir=LAYOUT["prediction_checkpoints"],
    metric_dir=LAYOUT["metric_partitions"],
    status_dir=LAYOUT["evaluation_status"],
    work_dir=work_dir,
    compute_spearman_primary=COMPUTE_SPEARMAN,
    compressed_npz=COMPRESSED_NPZ,
    verify_every_normalizer=True,
    sample_batch_size=SAMPLE_BATCH_SIZE,
    checkpoint_every=CHECKPOINT_EVERY,
    force=False,
    cleanup_work_on_success=CLEANUP_WORK_AFTER_FINALIZE,
    model_families=(),
    show_progress=False,
    finalize=True,
)
completion_path = LAYOUT["evaluation"] / "evaluation_completion.csv"
completion.to_csv(completion_path, index=False)
display(completion)

## Compact same-gene proxy table for notebook 8

Notebook 8 compares the three CCLE-trained same-gene proxies (raw RNA, CCLE-train z-score, CCLE-train linear regression) per cohort and protein, and keeps the best one per protein, cohort and metric (paper: *Input transformation and external validation*).
This cell writes them from the metric partitions just finalised, so no separate proxy notebook is needed.

Output: `PROJECT_ROOT/results_comparison_all_models_v4/proxy_models_v4/proxy_metrics_by_tissue_protein.csv`, the default path notebook 8 reads.

In [ ]:
PROXY_TABLE_DIR = PROJECT_ROOT / "results_comparison_all_models_v4" / "proxy_models_v4"
PROXY_TABLE_DIR.mkdir(parents=True, exist_ok=True)
proxy_tissue_protein = pv.proxy_metrics_by_tissue_protein(LAYOUT["metric_partitions"], MANIFEST_HASH)
proxy_tissue_protein_path = PROXY_TABLE_DIR / "proxy_metrics_by_tissue_protein.csv"
proxy_tissue_protein.to_csv(proxy_tissue_protein_path, index=False)
coverage = proxy_tissue_protein.groupby("display_model")["tissue"].nunique()
if (coverage != len(pv.COHORT_TABLE)).any():
    raise RuntimeError(f"Proxy table does not cover all cohorts for every proxy variant:\n{coverage}")
print("Wrote", proxy_tissue_protein_path, f"({len(proxy_tissue_protein):,} rows)")
display(coverage.rename("cohorts").to_frame())

## 6. Final completeness and provenance audit

In [ ]:
completion = pv.validate_evaluation_completion(
    LAYOUT["evaluation_status"],
    MANIFEST_HASH,
    pv.COHORT_TABLE["cohort"].tolist(),
)
if len(completion) != 20:
    raise RuntimeError(f"Expected 20 completed cohorts, found {len(completion)}")
if completion["model_failures"].sum() != 0:
    raise RuntimeError("One or more primary model predictions failed.")

expected_models = {"LinearProxy", "ElasticNet", "XGBoost"}
for models in completion["models"].astype(str):
    # Stored JSON lists may render as strings in the completion DataFrame.
    if not all(name in models for name in expected_models):
        raise RuntimeError(f"Incomplete finalized model set: {models}")

provenance = pv.notebook_provenance(
    "07_evaluation_paper_ready.ipynb",
    MANIFEST_HASH,
    {
        "manifest": str(MANIFEST_PATH.resolve()),
        "model_inventory_hash": manifest["model_inventory_hash"],
        "execution_mode": "separate_enet_xgb_progress_bars",
    },
    {
        "completion_table": str(completion_path.resolve()),
        "prediction_directory": str(LAYOUT["prediction_checkpoints"].resolve()),
        "metric_directory": str(LAYOUT["metric_partitions"].resolve()),
    },
)
pv.atomic_json_dump(provenance, LAYOUT["evaluation"] / "notebook07_provenance.json")

print("Completed cohorts:", len(completion))
print("ElasticNet run separately:", True)
print("XGBoost run separately:", True)
print("Progress bars:", "tqdm protein-wise")
print("Prediction checkpoints use long rows:", False)
print("External scTranslator:", "removed")